# IncidentCommanderEnv — SFT + GRPO Training

Self-contained Colab. **Run all** trains a Qwen-Coder LLM agent on the
IncidentCommander OpenEnv environment, end-to-end:

1. Install Unsloth + TRL + the env package
2. Load Qwen2.5-Coder-1.5B-Instruct in 4-bit with LoRA adapters
3. **SFT phase** — warm-start from senior-SRE behavioral-clone trajectories
4. Eval after SFT against random + base baselines
5. **GRPO phase** — fine-tune with the env's 6-component verifiable reward
6. Final eval — 4 conditions × 3 scenarios × 30 seeds = 360 episodes
7. Generate the 4 storytelling plots (reward curve, components, success bars, action dist)
8. Push the trained LoRA adapter to HuggingFace Hub

**Compute:** Qwen2.5-Coder-1.5B fits a free T4. For better code-fix
performance use A100 + bump model size to Qwen2.5-Coder-7B (instructions
in cell 3).

**Methodology mirrors the OpenEnv hackathon docs:** *"In many practical
cases, do a little SFT first, then RL"* — that's exactly this notebook.
Reward is RLVR (verifiable rewards), not a learned reward model. Six
independent reward components defeat single-axis hacking.

## 1. Install dependencies

Unsloth pins a specific transformers/peft/trl combo, so install in this order.

In [ ]:
%%capture
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.16" peft accelerate bitsandbytes datasets matplotlib huggingface_hub
!pip install fastapi pydantic uvicorn

## 2. Clone the IncidentCommanderEnv repo and import

Replace the URL with the user's actual fork.

In [ ]:
import os, sys, subprocess
REPO_URL = 'https://github.com/root4shreshth/incident-commander.git'
REPO_DIR = '/content/incident-commander'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR)
%cd {REPO_DIR}
!git pull --ff-only || true

# Smoke-import the env
from incident_commander_env.server.environment import IncidentCommanderEnv
env = IncidentCommanderEnv()
obs = env.reset(task_id='oom_crash', seed=42)
print('env imports OK; alert:', obs.alert[:80] if obs.alert else None)

## 3. Detect compute & choose model

Default: **Qwen2.5-Coder-1.5B-Instruct** (fits free T4). For A100, set
`MODEL_NAME = 'unsloth/Qwen2.5-Coder-7B-Instruct'`.

In [ ]:
import torch
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f'Device: {device_name}, VRAM: {vram_gb:.1f} GB')

# Auto-pick model based on VRAM
if vram_gb >= 35:
    MODEL_NAME = 'unsloth/Qwen2.5-Coder-7B-Instruct'
    MAX_SEQ_LEN = 4096
    BATCH_SIZE = 2
else:
    MODEL_NAME = 'unsloth/Qwen2.5-Coder-1.5B-Instruct'
    MAX_SEQ_LEN = 3072
    BATCH_SIZE = 2
print(f'Selected model: {MODEL_NAME} (max_seq_len={MAX_SEQ_LEN}, batch_size={BATCH_SIZE})')

## 4. Load model with Unsloth 4-bit + LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,            # auto
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)
print('Model + LoRA loaded.')

## 5. Build SFT dataset from senior-SRE trajectories

Replays each `IDEAL_TRAJECTORIES` entry under multiple seeds against the
live env to capture diverse (state, action) pairs.

In [ ]:
from training.datasets import build_sft_dataset, to_chat_messages, SYSTEM_PROMPT
from datasets import Dataset

rows = build_sft_dataset(n_seeds_per_family=8, seed_offset=0)
print(f'SFT rows: {len(rows)}')

def format_row(row):
    msgs = to_chat_messages(row)
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return {'text': text}

ds = Dataset.from_list(rows).map(format_row)
print(ds[0]['text'][:600])

## 6. SFT phase — 1 epoch warm-start

In [ ]:
from trl import SFTTrainer, SFTConfig
import os

sft_config = SFTConfig(
    output_dir='/content/sft-out',
    num_train_epochs=1,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy='no',
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    seed=42,
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field='text',
    bf16=False,
    fp16=True,
)

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    args=sft_config,
)
sft_history = sft_trainer.train()
model.save_pretrained('/content/sft-adapter')
tokenizer.save_pretrained('/content/sft-adapter')
print('SFT complete; adapter saved.')

## 7. Eval after SFT (random + base + sft)

Smaller eval (15 seeds × 3 families = 45 episodes) to confirm SFT helps
before sinking compute into GRPO.

In [ ]:
# ─── Suppress non-fatal generation warnings that repeat per call ───
import warnings, logging
warnings.filterwarnings("ignore", message=".*max_new_tokens.*max_length.*")
warnings.filterwarnings("ignore", category=FutureWarning,
                        module=r"transformers\.modeling_attn_mask_utils")
logging.getLogger("transformers").setLevel(logging.ERROR)

from training.eval_runner import evaluate, random_policy, hf_pipeline_policy
from training.datasets import SYSTEM_PROMPT
from transformers import GenerationConfig

FastLanguageModel.for_inference(model)

# Clear the conflicting default — Qwen sets max_length=32768 in its
# generation_config, and passing max_new_tokens alongside triggers a
# warning on every single generate() call. We only want max_new_tokens.
if hasattr(model, "generation_config") and model.generation_config is not None:
    model.generation_config.max_length = None

GEN_CFG = GenerationConfig(
    max_new_tokens=256,
    do_sample=False,
    temperature=0.0,
    pad_token_id=tokenizer.eos_token_id,
)

def hf_generate(messages, max_new=256):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, generation_config=GEN_CFG)
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    return full[len(text):] if full.startswith(text) else full

EVAL_FAMILIES = ["oom_crash", "db_pool_exhaustion", "bad_deployment_cascade"]
EVAL_SEEDS = list(range(2000, 2015))  # held-out range, no overlap with SFT seeds 0-7

report_random = evaluate("random", random_policy(rng_seed=99), EVAL_FAMILIES, EVAL_SEEDS, system_prompt=SYSTEM_PROMPT)
report_sft    = evaluate("sft",    hf_pipeline_policy(hf_generate), EVAL_FAMILIES, EVAL_SEEDS, system_prompt=SYSTEM_PROMPT)

print("After-SFT eval:")
for cond, rpt in [("random", report_random), ("sft", report_sft)]:
    print(f"
  {cond}:")
    for fam, stats in rpt.by_family.items():
        print(f"    {fam}: success={stats['success_rate']*100:>4.0f}%  "
              f"score={stats['avg_score']:.2f}  steps={stats['avg_steps_used']:.1f}")


## 8. GRPO phase — fine-tune with verifiable rewards

Switch back to training mode and use `GRPOTrainer` with our 6-component
reward function. The reward function logs each component to wandb (if
configured) for the killer storytelling plot.

In [ ]:
FastLanguageModel.for_training(model)

from trl import GRPOTrainer, GRPOConfig
from training.grpo_reward import grpo_reward_fn, reset_history, get_recent_breakdowns
from training.curriculum import Curriculum
from training.datasets import SYSTEM_PROMPT

reset_history()
curriculum = Curriculum(rng_seed=42)

GRPO_STEPS = 400

# Build a streaming-style dataset of prompts. Each row is (chat_messages, task_id, seed, difficulty).
def make_grpo_prompt_row(step):
    family, difficulty = curriculum.draw(step)
    seed = 5000 + step  # held-out training-prompt seed range
    env = IncidentCommanderEnv()
    reset_obs = env.reset(task_id=family, seed=seed, difficulty=difficulty)
    user_text = (f'INCIDENT ALERT:\n{reset_obs.message}\n\n'
                  'Begin investigation. What is your first action?')
    msgs = [{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':user_text}]
    return {'prompt': msgs, 'task_id': family, 'seed': seed, 'difficulty': difficulty}

from datasets import Dataset as HFDataset
grpo_ds = HFDataset.from_list([make_grpo_prompt_row(i) for i in range(GRPO_STEPS)])

grpo_config = GRPOConfig(
    output_dir='/content/grpo-out',
    learning_rate=5e-6,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_prompt_length=2048,
    max_completion_length=384,
    num_train_epochs=1,
    logging_steps=5,
    save_strategy='no',
    seed=42,
    bf16=False,
    fp16=True,
    beta=0.04,  # KL coefficient
)

grpo_trainer = GRPOTrainer(
    model=model,
    reward_funcs=[grpo_reward_fn],
    args=grpo_config,
    train_dataset=grpo_ds,
    processing_class=tokenizer,
)
grpo_history = grpo_trainer.train()
model.save_pretrained('/content/grpo-adapter')
tokenizer.save_pretrained('/content/grpo-adapter')
print('GRPO complete; adapter saved.')

## 9. Final eval — 4 conditions × 3 families × 30 seeds = 360 episodes

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*max_new_tokens.*max_length.*")

FastLanguageModel.for_inference(model)
if hasattr(model, "generation_config") and model.generation_config is not None:
    model.generation_config.max_length = None

FINAL_SEEDS = list(range(3000, 3030))  # 30 held-out seeds, no overlap with SFT or GRPO
report_grpo         = evaluate("sft+grpo", hf_pipeline_policy(hf_generate), EVAL_FAMILIES, FINAL_SEEDS, system_prompt=SYSTEM_PROMPT)
report_random_final = evaluate("random",   random_policy(rng_seed=12345),    EVAL_FAMILIES, FINAL_SEEDS, system_prompt=SYSTEM_PROMPT)

# Note: 'base' eval requires the un-trained adapter; we approximate via the
# random-policy floor. The hackathon submission framing is "random vs SFT vs
# SFT+GRPO" — the trained conditions are the meaningful comparison.

print("Final results:")
for name, rpt in [("random", report_random_final), ("sft+grpo", report_grpo)]:
    print(f"
  {name}:")
    for fam, stats in rpt.by_family.items():
        print(f"    {fam}: success={stats['success_rate']*100:>4.0f}%  "
              f"score={stats['avg_score']:.2f}  steps={stats['avg_steps_used']:.1f}")


## 10. Generate plots → `results/`

In [ ]:
import os
os.makedirs('/content/results', exist_ok=True)

from training.plots import (
    make_reward_curve, make_reward_components,
    make_success_bars, make_action_distribution, save_figure,
)

# Reward curve (if GRPO logged log_history)
if hasattr(grpo_trainer, 'state') and grpo_trainer.state.log_history:
    log = grpo_trainer.state.log_history
    rewards = [entry.get('reward', 0) for entry in log if 'reward' in entry]
    steps_log = list(range(len(rewards)))
    if rewards:
        save_figure(make_reward_curve(steps_log, rewards), '/content/results/reward_curve.png')

# Reward components (from grpo_reward sidecar)
bds = get_recent_breakdowns()
if bds:
    components = {k: [bd.to_dict()[k] for bd in bds] for k in ['diagnostic','correct_op','resolution','format','efficiency','penalty']}
    # Smooth into a moving average for a cleaner trend line
    W = 32
    smoothed = {k: [sum(v[max(0,i-W):i+1])/min(i+1,W) for i in range(len(v))] for k, v in components.items()}
    save_figure(
        make_reward_components(list(range(len(bds))), smoothed),
        '/content/results/reward_components.png',
    )

# Success bars
save_figure(
    make_success_bars(
        {'random': report_random_final.by_family,
         'sft':    report_sft.by_family,
         'sft+grpo': report_grpo.by_family},
        EVAL_FAMILIES,
    ),
    '/content/results/success_rates.png',
)

# Action distribution
actions_per_cond = {}
for cond, rpt in [('random', report_random_final), ('sft', report_sft), ('sft+grpo', report_grpo)]:
    counter = {}
    for fam_stats in rpt.by_family.values():
        for action, n in fam_stats['action_distribution'].items():
            counter[action] = counter.get(action, 0) + n
    actions_per_cond[cond] = counter
save_figure(
    make_action_distribution(actions_per_cond),
    '/content/results/action_distribution.png',
)

print('Plots saved:')
for f in os.listdir('/content/results'):
    print(f'  /content/results/{f}')

## 11. Save eval JSON + push adapter to HuggingFace Hub

Set `HF_USER` and run `huggingface-cli login` first if you want to push.

In [ ]:
import json
summary = {
    'model': MODEL_NAME,
    'eval_seeds': FINAL_SEEDS,
    'families': EVAL_FAMILIES,
    'conditions': {
        'random':   report_random_final.by_family,
        'sft':      report_sft.by_family,
        'sft+grpo': report_grpo.by_family,
    },
}
with open('/content/results/eval_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved /content/results/eval_summary.json')

# Optional: push to HF Hub
HF_USER = os.environ.get('HF_USER', '')
if HF_USER:
    from huggingface_hub import login, create_repo
    if 'HF_TOKEN' in os.environ:
        login(os.environ['HF_TOKEN'])
    repo_id = f'{HF_USER}/incident-commander-grpo'
    create_repo(repo_id, exist_ok=True)
    model.push_to_hub(repo_id)
    tokenizer.push_to_hub(repo_id)
    print(f'Pushed to {repo_id}')
else:
    print('Set HF_USER + HF_TOKEN env vars to enable Hub upload.')

## 12. Summary table

Drop this into the README's Results section.

In [ ]:
print('| Condition | OOM | DB Pool | Bad Deploy | Avg |')
print('|---|---:|---:|---:|---:|')
for cond, rpt in [('Random',   report_random_final),
                  ('SFT',      report_sft),
                  ('SFT+GRPO', report_grpo)]:
    rates = [rpt.by_family[fam]['success_rate'] for fam in EVAL_FAMILIES]
    avg = sum(rates) / len(rates)
    print(f'| {cond} | {rates[0]*100:>3.0f}% | {rates[1]*100:>3.0f}% | {rates[2]*100:>3.0f}% | **{avg*100:>3.0f}%** |')